In [1]:
import numpy as np
from scipy.sparse import csr_matrix
from tqdm import tqdm
import numpy as np
import scipy.sparse as sp
from scipy.sparse import diags
from scipy.linalg import lu_factor, lu_solve
import matplotlib.pyplot as plt
from functools import lru_cache
from scipy.linalg import solve
import os

In [2]:
def beta_matrix(w, d, t, e, m):
    dim = 2 * m

    w = w[:,None,None]

    base_val = (w + 1j*d)

    base = base_val * np.eye(2*m)[None,:,:]


    idy = np.arange(0,2*m-1,1)

    base[:,idy,idy+1] = t
    base[:,idy+1,idy] = t

    idx = np.arange(0,m,2)

    base[:,idx,2*m-1-idx] = t
    base[:,2*m-1-idx,idx] = t


    return base 

In [3]:
def T1_matrix(t, m):
    dim = 2 * m
    T = np.zeros((dim, dim), dtype=np.complex128)

    n = np.arange(1, (m - 1)//2 + 1)
    i = 2*n - 1           # to 0-based
    j = 2*m - 2*n

    T[i, j] = t
    return T


In [5]:

def leads_vectorized(w_vals, d, t, e, n, tol=1e-6, max_iter=400000):
    B = len(w_vals)
    unit_batch = beta_matrix(w_vals, d, t, e, n)
    
    # g = inv(unit) using solve (much faster than LU)
    eye_dim = np.eye(unit_batch.shape[-1])
    g = np.stack([solve(unit_batch[i], eye_dim) for i in range(B)])
    
    G = g.copy()
    hopp = T1_matrix(t, n)
    hoppT = hopp.T
    dim = g.shape[-1]
    iden = np.eye(dim)[None,:,:]

    diff = np.inf
    count = 0
    pbar = tqdm(total=max_iter, desc="Dyson iteration", leave=True)

    while diff > tol and count < max_iter:
        A = iden - g @ hoppT @ G @ hopp

        G_new = np.stack([solve(A[i], g[i]) for i in range(B)])

        diff = np.max(np.abs(G_new - G))
        G = G_new
        count += 1
        
        pbar.update(1)
        pbar.set_postfix({"diff": diff})

    pbar.close()
    return G, count


In [6]:

def rho_matrix(t, m):
    
    dim = 2 * m
    rho = np.zeros((dim, dim), dtype=complex)
    for n in range(1, (m - 1) // 2 + 1):
        idx = 2 * n - 1
        rho[idx, idx] = t
    return rho
print(np.abs(rho_matrix(1,7)))

[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [7]:
def compute(n):
    w_vals = np.arange(0, 3, 0.01)

    G_all, iterations = leads_vectorized(
        w_vals=w_vals,
        d=1e-5,
        t=1,
        e=0,
        n=n
    )


    return G_all, iterations

In [ ]:
g_7 = compute(7)

In [ ]:
import os

In [8]:
for l in range(5, 15):
    dir_path = os.path.expanduser(f'~/Desktop/backup/agnr/size_{l}')
    os.makedirs(dir_path, exist_ok=True)
    print(f"working on : ====> {l}" )
    G_all, iters = compute(l)

    fn = os.path.join(dir_path, f'leads_{l}.npy')
    np.save(fn, G_all)


working on : ====> 5


Dyson iteration:  90%|█████████ | 361556/400000 [1:56:10<12:21, 51.87it/s, diff=1e-6]      


working on : ====> 6


Dyson iteration:  92%|█████████▏| 367062/400000 [2:01:14<10:52, 50.46it/s, diff=1e-6]      


working on : ====> 7


...

working on : ====> 11


Dyson iteration:  89%|████████▉ | 356906/400000 [3:13:11<23:19, 30.79it/s, diff=1e-6]       


working on : ====> 12


Dyson iteration:   3%|▎         | 12866/400000 [13:58<5:51:58, 18.33it/s, diff=32.6]  

KeyboardInterrupt: 

In [ ]:
def connection(t,m):
    idx = np.arange(2,m,2)
    base = np.zeros((2*m,2*m),dtype=np.complex64)

    base[2*m - idx,idx - 1] = t 

    return base

In [ ]:
np.abs(connection(1,7)).T[1,12]

In [ ]:
def unitcell(w, d, t, e, m):
    dim = 2 * m

    base_val = (w + 1j*d)

    base = base_val * np.eye(2*m)

    idy = np.arange(0,2*m-1,1)

    base[idy,idy+1] = t
    base[idy+1,idy] = t

    idx = np.arange(0,m,2)

    base[idx,2*m-1-idx] = t
    base[2*m-1-idx,idx] = t


    return base 

In [ ]:
g_7 = compute(7)

np.save(os.path.expanduser(r"c:\Users\mukim\Downloads\transmissions\leads\agnr_7.npy"),g_7)

In [ ]:
path = os.getcwd()
print(path)

In [ ]:


# ----------------------------------------------------------------------
# 1) Precomputed global cache of device_combs per width
# ----------------------------------------------------------------------
DEVICE_COMBS = {}


def _get_device_combs(width: int) -> np.ndarray:
    """
    Return (global cached) device combinations for a given width.
    Each row is (i, j) with i in [0..99] and j in [0..width-1].
    """
    width = int(2*width)
    if width not in DEVICE_COMBS:
        # Build once, store once
        DEVICE_COMBS[width] = np.stack(
            np.meshgrid(np.arange(100), np.arange(width), indexing="ij"),
            axis=-1
        ).reshape(-1, 2)
    return DEVICE_COMBS[width]


# ----------------------------------------------------------------------
# 2) Cached selection of N random impurity sites for a given config
# ----------------------------------------------------------------------
@lru_cache(maxsize=2048)
def chosen_for_config(n: int, width: int, config: int) -> np.ndarray:
    """
    Return n rows selected deterministically by given config.
    Completely cached.
    """
    n = int(n)
    width = int(width)
    config = int(config)

    device_combs = _get_device_combs(width)

    rng = np.random.RandomState(config)
    idx = rng.choice(len(device_combs), size=n, replace=False)
    return device_combs[idx]


# ----------------------------------------------------------------------
# 3) Factory: returns function(seed) → chosen impurity rows
# ----------------------------------------------------------------------
def possible_combs(n: int, width: int):
    n = int(n)
    width = int(width)

    def combs_for_seed(seed: int):
        return chosen_for_config(n, width, seed)

    return combs_for_seed


# ----------------------------------------------------------------------
# 4) Device builder
# ----------------------------------------------------------------------
def unidevice(w, d, t, e, size, config, n, numberofunitcell, combs_fn=None):
    """
    Build a device Hamiltonian unit with impurities inserted at positions
    determined by 'config'. Uses vectorised assignments where possible.
    """

    size = int(size)

    
    if combs_fn is None:
        combs_fn = possible_combs(int(n), int(size))

    # Chosen impurity coordinates
    imps = combs_fn(int(config))
    x = imps[:, 0]
    y = imps[:, 1]

    z = int(numberofunitcell)

    # Base Hamiltonian (already cached inside your unitcell_leads)
    mat = unitcell(w, d, t, e, int(size))

    # Identify impurities in unitcell z
    mask = (x == z)
    if not np.any(mask):
        return mat  # fast path return

    imp_indices = y[mask]

    # Vectorised diagonal modification
    diag_val = (w + 1j*d - 0.5)
    mat[imp_indices, imp_indices] = diag_val

    return mat


In [ ]:
g_7 = np.load(os.path.expanduser(f'~/Desktop/backup/agnr/size_{7}/leads_{7}.npy'))

In [ ]:
def device_transmission(w, d, t, e, size, config, concentration, x):
    ene = int(w * 100)
    m = size
    dim = 2 * m
    I = np.eye(dim, dtype=complex)

    # 1. Lead Surface Green's Functions:
    left = g_7[0][ene]                   # Left lead surface Green's function
    #P = np.eye(dim)[::-1]
    #right = P @ left @ P                  # Right lead surface Green's function (g_R = P g_L P)

    # 2. Hopping Matrices:
    #T1 = T1_matrix(t, m)     
    tin = T1_matrix(t, m)
    tin_d = tin.T             # Forward hopping T
    #Tcon = connection(t, m)               # Backward hopping T^\dagger
    rho = rho_matrix(t, m)                # Contact operator \[Rho]

    # 3. Device Propagation (100 Unit Cells):
    combs_fn = possible_combs(concentration, size)
    g_new = left

    for i in range(100):
        unit_i = unidevice(w, d, t, e, size, config, concentration, i, combs_fn=combs_fn)
        gd = np.linalg.inv(unit_i)
        G = np.linalg.solve(I - gd @ tin_d @ g_new @ tin, gd)
        g_new = G

    left_device = g_new

    # 4. Connected Interface Green's Functions using Mathematica's rho matrix:
    IL = np.linalg.solve(I - left_device @ rho @ left @ rho, left_device)
    IR = np.linalg.solve(I - left @ rho @ left_device @ rho, left)

    # 5. Spectral Functions & Non-local Cross Green's Function:
    gdd = IL - IL.conj().T
    grr = IR - IR.conj().T

    Gnonlocal = left @ rho @ IR
    GNON = Gnonlocal - Gnonlocal.conj().T

    # 6. Mathematica Trace Calculation:
    term1 = gdd @ rho @ grr @ rho
    term2 = rho @ GNON @ rho @ GNON

    tr1 = np.abs(np.trace(term1 - term2))

    return -np.imag(IL[x, x]), -np.imag(IR[x, x]), np.abs(tr1)

In [ ]:

for x in range(13,14):
    dos1,dos2,tran =[],[],[] 
    for w in np.arange(0,3,0.01):
        a,b,c = device_transmission(w,1e-4,1,0,7,1,0,x) 
        dos1.append(a)
        dos2.append(b)
        tran.append(c)

    plt.plot(dos1, label = f'{x}',linestyle='--')
    plt.plot(dos2, label = f'{x}')
    plt.plot(tran, label = f'{x}-trans')
    plt.ylim(0,3.2)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.show()

In [ ]:
a,b = [device_transmission(0,1e-3,1,0,7,1,0,1) for w in np.arange(0,3,0.01)]

In [ ]:
def device_transmission(w, d, t, e, size, config, concentration, x=):
    ene = int(w * 100)
    m = size
    dim = 2 * m

    left = g_7[ene]                      # left lead surface Green's function g_L
    P = np.eye(dim)[::-1]
    right = P @ left @ P                  # right lead surface Green's function g_R = P g_L P

    T = T1_matrix(t, m)                   # hopping matrix T (cell i to i+1)
    TT = connection(t, m)                 # hopping matrix T^T (cell i+1 to i)

    I = np.eye(dim, dtype=complex)

    # 1. Lead broadening matrices:
    sigma_L = TT @ left @ T
    gamma_L = 1j * (sigma_L - sigma_L.conj().T)

    sigma_R = T @ right @ TT
    gamma_R = 1j * (sigma_R - sigma_R.conj().T)

    # 2. Device propagation through 100 unit cells:
    combs_fn = possible_combs(concentration, size)
    unit0 = unidevice(w, d, t, e, size, config, concentration, 0, combs_fn=combs_fn)
    gd0 = np.linalg.inv(unit0)

    g_surf = np.linalg.solve(I - gd0 @ TT @ left @ T, gd0)
    G_cross = g_surf.copy()

    for i in range(1, 100):
        unit_i = unidevice(w, d, t, e, size, config, concentration, i, combs_fn=combs_fn)
        gd = np.linalg.inv(unit_i)
        g_surf_next = np.linalg.solve(I - gd @ TT @ g_surf @ T, gd)
        G_cross = G_cross @ T @ g_surf_next
        g_surf = g_surf_next

    # Full retarded cross Green's function G_{0, 99}^R from cell 0 to cell 99:
    G_0_99 = G_cross @ np.linalg.solve(I - T @ right @ TT @ g_surf, I)

    if x is not None:
        IL = np.linalg.solve(I - g_surf @ T @ right @ TT, g_surf)
        return -np.imag(IL[x, x]) / np.pi  # LDOS at site x

    # Fisher-Lee Transmission: T(E) = Tr[ gamma_L @ G_0_99 @ gamma_R @ G_0_99^\dagger ]
    tr1 = np.real(np.trace(gamma_L @ G_0_99 @ gamma_R @ G_0_99.conj().T))

    return np.abs(tr1)

pristine = np.array([device_transmission(w,0.0001,1,0,7,0,0) for w in np.arange(0,3,0.01)])
plt.plot(pristine)
plt.ylim(-0.2,3)

In [ ]:
device_transmission(0,1e-3,1,0,7,1,10)

In [ ]:
w = np.arange(0,3,0.01)
"""  device_transmission(w, d, t, e,size,config,concentration)"""


def transmission(config, conc):
    trans = [device_transmission(i, 0.0001, 1, 0,7,config,conc) for i in np.arange(0,3,0.01)]

    return trans


In [ ]:
pris = [device_transmission(i, 0.0001, 1, 0,7,1,0) for i in np.arange(0,3,0.01)]

In [ ]:
import h5py
import numpy as np
import os
from tqdm import tqdm
from multiprocessing import Pool, cpu_count


def run_transmission(args):
    config, conc = args
    return transmission(config, conc)


def main():
    concs = np.arange(1, 50, 2)
    nc = len(concs)

    nconfigs = 10000
    nE = 300
    nF = 1

    outdir = os.path.expanduser('~/Desktop/backup/agnr/size_21')
    os.makedirs(outdir, exist_ok=True)
    h5_path = f"{outdir}/all_transmissions_mp.h5"

    NPROC = max(1, cpu_count() - 1)
    print(f"Using {NPROC} CPU cores…")

    with h5py.File(h5_path, "w") as f:

        dset = f.create_dataset(
            "transmissions",
            shape=(nc, nconfigs, nE, nF),
            dtype=np.float64,
            compression="gzip",
            compression_opts=4,
            chunks=(1, 500, nE, nF)
        )

        for i, conc in enumerate(tqdm(concs, desc="Concentration sweep")):

            args = [(cfg, conc) for cfg in range(nconfigs)]

            batch_list = []
            with Pool(NPROC) as p:
                for result in tqdm(
                    p.imap(run_transmission, args, chunksize=20),
                    total=nconfigs,
                    desc=f"configs conc={conc}",
                    leave=False,
                ):
                    batch_list.append(result)

            batch = np.asarray(batch_list).reshape(nconfigs, nE, nF)
            dset[i] = batch


if __name__ == "__main__":
    main()
